In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e11/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e11/train.csv
/kaggle/input/competitions/playground-series-s6e11/test.csv


In [2]:
# Step 1: load the tables and look at what is inside
d = "/kaggle/input/competitions/playground-series-s6e11/"
train = pd.read_csv(d + "train.csv")
test = pd.read_csv(d + "test.csv")
print("rows, columns:", train.shape, test.shape)
print("share of fraud rows:", train["is_fraud"].mean())
print("missing values in train:", train.isna().sum().sum())
print(train.dtypes.value_counts())
pd.set_option("display.max_rows", 100)
train.head(3).T

rows, columns: (699960, 26) (299984, 25)
share of fraud rows: 0.016699525687182123
missing values in train: 0
int64      8
float64    7
bool       6
object     5
Name: count, dtype: int64


,0,1,2
id,0,1,2
amount_usd,61.16,24.83,6.36
hours_since_last_txn,9.77,1.54,1.2
txn_count_last_24h,1,3,4
distance_from_home_km,17.78,13.06,8.17
card_age_months,48,38,50
customer_age,56,28,55
account_balance_usd,1269.35,3233.0,1046.88
cvv_retry_count,0,0,0
velocity_score,5.5,33.4,35.8


In [3]:
# Step 2: Feature Engineering + 5-fold LightGBM
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def make_features(df):
    out = df.copy()
    out["amount_to_balance"] = out["amount_usd"] / (out["account_balance_usd"] + 1.0)
    out["balance_after_txn"] = out["account_balance_usd"] - out["amount_usd"]
    out["txn_burst_rate"] = out["txn_count_last_24h"] / (out["hours_since_last_txn"] + 0.1)
    flag_cols = [
        "is_foreign_transaction", "is_new_merchant", "used_vpn",
        "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
    ]
    out["risk_flag_count"] = out[flag_cols].sum(axis=1)
    out["amount_x_merchant_risk"] = out["amount_usd"] * out["merchant_risk_score"]
    return out

train_fe = make_features(train)
test_fe = make_features(test)

text_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
for c in text_cols:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = pd.Categorical(test_fe[c], categories=train_fe[c].cat.categories)

X = train_fe.drop(columns=["id", "is_fraud"])
y = train_fe["is_fraud"]
X_test = test_fe.drop(columns=["id"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

if "oof_preds" not in globals():
    oof_preds = np.zeros(len(train_fe))
    test_preds = np.zeros(len(test_fe))
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        model = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=31,
            min_child_samples=100, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=fold, verbose=-1, n_jobs=-1,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)], eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        oof_preds[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / 5

lgb_oof = oof_preds.copy()
lgb_test = test_preds.copy()
print("LightGBM 5-fold OOF AUC:", round(roc_auc_score(y, lgb_oof), 5))

LightGBM 5-fold OOF AUC: 0.94644


In [4]:
# Step 3: 5-fold XGBoost
import xgboost as xgb

if "xgb_oof" not in globals():
    xgb_oof = np.zeros(len(train_fe))
    xgb_test = np.zeros(len(test_fe))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        xgb_model = xgb.XGBClassifier(
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=100,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            enable_categorical=True,
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=fold,
            n_jobs=-1,
        )
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        xgb_oof[va_idx] = xgb_model.predict_proba(X_va)[:, 1]
        xgb_test += xgb_model.predict_proba(X_test)[:, 1] / 5

print("XGBoost 5-fold OOF AUC:", round(roc_auc_score(y, xgb_oof), 5))

XGBoost 5-fold OOF AUC: 0.94561


In [5]:
# Step 4: 5-fold CatBoost + 3-Model Blend (LightGBM + XGBoost + CatBoost)
from catboost import CatBoostClassifier

if "cb_oof" not in globals():
    X_cb = X.copy()
    X_test_cb = X_test.copy()
    for c in text_cols:
        X_cb[c] = X_cb[c].cat.codes
        X_test_cb[c] = X_test_cb[c].cat.codes

    cb_oof = np.zeros(len(train_fe))
    cb_test = np.zeros(len(test_fe))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y)):
        X_tr, y_tr = X_cb.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_cb.iloc[va_idx], y.iloc[va_idx]

        cb_model = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.06,
            depth=6,
            cat_features=text_cols,
            one_hot_max_size=25,
            eval_metric="AUC",
            random_seed=fold,
            thread_count=-1,
            verbose=False,
        )
        cb_model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=50,
            verbose=False,
        )

        cb_oof[va_idx] = cb_model.predict_proba(X_va)[:, 1]
        cb_test += cb_model.predict_proba(X_test_cb)[:, 1] / 5

blend3_oof = 0.5 * lgb_oof + 0.1 * xgb_oof + 0.4 * cb_oof
blend3_test = 0.5 * lgb_test + 0.1 * xgb_test + 0.4 * cb_test
print("3-Way Blend (0.5 LGB + 0.1 XGB + 0.4 CB) OOF AUC:", round(roc_auc_score(y, blend3_oof), 5))

3-Way Blend (0.5 LGB + 0.1 XGB + 0.4 CB) OOF AUC: 0.94678


In [6]:
# Step 5: write the best 3-Model Blend (0.5 LGB + 0.1 XGB + 0.4 CB = 0.94678 OOF AUC) to submission.csv
sub = pd.DataFrame({"id": test["id"], "is_fraud": blend3_test})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,is_fraud
0,699960,0.000276
1,699961,0.000163
2,699962,0.086757
3,699963,0.001527
4,699964,0.001414
